In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report

# Charger les données
df = pd.read_csv("preparation_data.csv")

# Imputation des valeurs manquantes
imputer = SimpleImputer(strategy="most_frequent")
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# Regrouper les États en clusters avec KMeans
for col in ["State", "BankState"]:
    state_kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)  # 10 clusters (modifiable)
    df_imputed[col + "_Cluster"] = state_kmeans.fit_predict(df_imputed[[col]])

# Supprimer les colonnes originales State et BankState
df_imputed = df_imputed.drop(columns=["State", "BankState"])

# Encodage des variables binaires
binary_cols = ["RevLineCr", "LowDoc"]
for col in binary_cols:
    df_imputed[col] = LabelEncoder().fit_transform(df_imputed[col])

# Séparation des variables explicatives et cible
X = df_imputed.drop(columns=["MIS_Status"])
y = df_imputed["MIS_Status"]

# Détection et suppression des valeurs aberrantes avec IsolationForest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(X)
X_cleaned = X[outliers == 1]
y_cleaned = y[outliers == 1]

# Division en train et test
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned)

# Normalisation des données
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# Apprentissage du modèle
model = LogisticRegression(max_iter=6000)
model.fit(X_train_scaled, y_train)

# Prédiction
y_pred = model.predict(X_test_scaled)

# Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


ValueError: could not convert string to float: 'IN'

In [4]:
df.isna().sum()

Term                     0
FranchiseCode            0
State                   14
BankState             1566
NAICS_2                  0
NoEmp                    0
NewExist               136
RetainedJob              0
CreateJob                0
UrbanRural               0
RevLineCr            19854
ApprovalFY               0
DisbursementGross        0
GrAppv                   0
LowDoc                6007
MIS_Status            1997
dtype: int64